# Lab 8 · Chuỗi thời gian: quý, cửa sổ trượt và so cùng kỳ

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 8**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook Bài 8 làm việc theo **tháng**. Lab này chuyển sang hai độ phân giải khác:
**ngày** (cửa sổ trượt 7 ngày) và **quý**, sau đó tính phép so cùng kỳ.

## Cách làm việc trong buổi lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau buổi lab, bạn sẽ:

1. Cắt lát thời gian bằng chuỗi (`.loc["2019"]`) và đối chiếu với con số đã biết.
2. Tổng hợp theo quý (`resample`); nhận diện và loại kỳ chưa trọn ở cấp quý.
3. Làm mượt chuỗi ngày bằng cửa sổ trượt (`rolling`) và đọc điểm cao/thấp.
4. Tính so với cùng kỳ năm trước và phân biệt với so kỳ liền trước.

## Phần 0 · Khởi động (~8 phút)

In [ ]:
import pandas as pd

# W1 — nạp dữ liệu, loại ngày sau mốc chụp, đưa date vào chỉ mục và sắp xếp
SNAPSHOT = pd.Timestamp("2026-06-29")
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/reviews.csv")
rv_raw = pd.read_csv(URL, parse_dates=["date"])
so_dong_sau_moc = (rv_raw["date"] > SNAPSHOT).sum()
rv = rv_raw.loc[rv_raw["date"] <= SNAPSHOT].copy()
# TODO: tạo r = rv với chỉ mục là date, đã sort_index
r = ...

# --- Ô kiểm tra ---
assert so_dong_sau_moc == 204
assert str(type(r.index)).endswith("DatetimeIndex'>")
assert r.index.is_monotonic_increasing
print(f"W1 ổn — đã loại {so_dong_sau_moc} dòng sau mốc chụp dữ liệu; trục thời gian sẵn sàng.")

In [ ]:
# W2 — cắt lát bằng chuỗi + đối chiếu số đã biết
# Ở lab 2 bạn đã đếm bằng dict: 2019 có 24.299 đánh giá, 2021 có 23.926.
# TODO: đếm lại bằng cắt lát chuỗi trên r
n_2019 = ...
n_2021 = ...

# --- Ô kiểm tra ---
assert n_2019 == 24299 and n_2021 == 23926
print(f"Khớp lab 2. Đến 2021 thị trường đã hồi {n_2021 / n_2019:.0%} mức 2019.")

## Phần 1 · Nhìn theo quý (~25 phút)

### Bước 1 · Tổng hợp theo quý và nhận diện kỳ chưa trọn

In [ ]:
# TODO: đếm đánh giá theo quý (mã tần suất "QE")
quy = ...

# --- Ô kiểm tra ---
assert quy.loc["2026-03-31"] == 68395     # quý 1/2026
assert quy.loc["2026-06-30"] == 60064     # quý 2 chưa trọn
quy.tail(4)

Sau khi loại 204 dòng sau mốc chụp dữ liệu, quý cuối là **Q2/2026** với 60.064 đánh giá.
Quý này vẫn chưa trọn vì thiếu ngày 30/06. Kỳ gần đủ thường khó nhận ra nếu chỉ nhìn
vào số lượng; luôn phải đối chiếu nhãn kỳ với ngày chụp dữ liệu.

In [ ]:
# TODO: tạo quy_day_du — bỏ quý chưa trọn cuối cùng, rồi tìm quý cao nhất và thấp nhất
#        trong giai đoạn 2019–2021 (dùng .loc cắt lát trước khi idxmin)
quy_day_du = ...
quy_dinh = ...          # nhãn thời gian của quý cao nhất toàn lịch sử (idxmax)
quy_day_covid = ...     # nhãn của quý THẤP nhất trong 2019–2021 (idxmin trên lát cắt)

# --- Ô kiểm tra ---
assert len(quy_day_du) == len(quy) - 1
assert str(quy_dinh)[:10] == "2026-03-31"
assert str(quy_day_covid)[:10] == "2020-06-30" and quy.loc[quy_day_covid] == 1022
print(f"Cao nhất: Q1/2026 ({quy_day_du.max():,}) — thấp nhất giai đoạn 2019–2021: Q2/2020 ({quy.loc[quy_day_covid]:,}).")

Q2/2020 có 1.022 đánh giá, bằng khoảng 1/67 quý cao nhất. Khi báo cáo giá trị cực trị,
cần ghi cả giá trị và mốc thời gian tương ứng.

## Phần 2 · Làm mượt chuỗi ngày bằng cửa sổ trượt (~22 phút)

In [ ]:
# TODO: đếm đánh giá theo ngày của riêng năm 2025 (cắt lát trước, tổng hợp bằng resample("D") sau)
ngay_2025 = ...

# TODO: làm mượt bằng cửa sổ trượt 7 ngày, căn giữa (rolling(7, center=True).mean())
tb7 = ...

# --- Ô kiểm tra ---
assert len(ngay_2025) == 365
assert ngay_2025.max() == 1460
assert str(tb7.idxmax())[:10] == "2025-11-23"
print(f"Mốc cao nhất năm 2025 theo trung bình 7 ngày: {tb7.idxmax():%d/%m/%Y} — {tb7.max():.0f} đánh giá/ngày.")

In [ ]:
# Vẽ nhanh để quan sát cửa sổ trượt (mã cho sẵn — Bài 12 học kỹ hơn)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(ngay_2025.index, ngay_2025.values, color="#bbb", lw=0.8, label="theo ngày")
ax.plot(tb7.index, tb7.values, color="#1E93AB", lw=2, label="trượt 7 ngày")
ax.legend(frameon=False)
ax.set_title("Số đánh giá theo ngày năm 2025 và trung bình trượt 7 ngày")
plt.show()

Đường theo ngày có chu kỳ tuần rõ rệt; cửa sổ 7 ngày làm giảm dao động này để xu hướng
dài hơn dễ quan sát. Độ rộng cửa sổ nên được chọn theo chu kỳ cần làm mượt.

## Phần 3 · So cùng kỳ — con số báo cáo được (~15 phút)

Tháng 5/2026 có 22.296 đánh giá. Mức này cần được đặt cạnh một mốc so sánh phù hợp.

In [ ]:
thang = r.resample("ME").size()

# TODO: tính 2 phép so cho tháng 5/2026:
so_ky_truoc = ...    # % thay đổi so với THÁNG 4/2026 (dùng .loc hai nhãn, tự tính %)
so_cung_ky = ...     # % thay đổi so với THÁNG 5/2025

# --- Ô kiểm tra ---
assert round(so_ky_truoc, 1) == 3.0
assert round(so_cung_ky, 1) == 53.7
print(f"T5/2026: +{so_ky_truoc:.0f}% so tháng trước · +{so_cung_ky:.0f}% so cùng kỳ.")

Hai con số trả lời hai câu hỏi khác nhau: +3% so với tháng trước mô tả thay đổi ngắn hạn,
còn +54% so với cùng kỳ đặt tháng 5 cạnh cùng vị trí trong chu kỳ năm. Với dữ liệu có
mùa vụ, phép so cùng kỳ thường phù hợp hơn để đánh giá thay đổi dài hạn.

## Phần 4 · Bài tự làm ✅ mở

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại yêu cầu đã gửi cho AI và cách kiểm chứng.

### Tự làm 1 · Điều tra ngày kỷ lục

Tìm **ngày có nhiều đánh giá nhất toàn bộ dữ liệu** (tổng hợp theo ngày bằng `resample("D")`). Bạn sẽ thấy
một ngày giữa tháng 3/2026 với hơn 2.000 đánh giá, cao hơn rõ rệt các ngày quanh nó và là
**thứ Hai**. Hãy: (1) in số đánh giá của 5 ngày quanh nó; (2) tra thử tuần đó ở Santiago có
sự kiện gì (gợi ý: một lễ hội âm nhạc lớn thường diễn ra giữa tháng 3); (3) viết 2 câu
kết luận kiểu báo cáo — có dè chừng đúng mực (đây là *một* cách giải thích, chưa kiểm chứng
độc lập).

### Tự làm 2 · Thời gian từ đánh giá gần nhất

Với bảng `listings` (bản rút gọn), trước hết loại `last_review` sau mốc chụp dữ liệu rồi tính số ngày
từ `last_review` đến 29/06/2026. Bao nhiêu chỗ ở có đánh giá gần nhất **cách đây hơn một năm**
(>365 ngày)? Đề xuất một cờ dữ liệu cho nhóm lâu không có đánh giá mới.

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Cắt lát chuỗi + đối chiếu số lab 2 | kiểm chứng chéo giữa các công cụ |
| Loại ngày sau mốc chụp dữ liệu và quý chưa trọn | mọi biểu đồ thời gian của bài tập lớn |
| Cửa sổ trượt 7 ngày (`rolling`) làm giảm chu kỳ tuần | trực quan hoá chuỗi thời gian (Bài 12) |
| So kỳ trước và so cùng kỳ | chọn mốc so sánh phù hợp |

Tuần tiếp theo là **thi giữa kỳ**. Sau giữa kỳ, Bài 10 tiếp tục với làm sạch dữ liệu có cấu trúc.